STAMMING - ALGORITMI PORTER E SNOWBALL

Lo stemming è una tecnica di preprocessing NLP che cerca di ridurre parole diverse a una stessa radice, chiamata stem.
Ordinare, ordinato, ordina, ordini, ordinando
possono essere ridotte a qualcosa come:
ordin
Quindi lo stemming non cerca necessariamente una parola italiana corretta, ma cerca una forma comune utile a raggruppare parole simili.
Questo è il punto importante:
connessione, connesso, connettere possono essere ridotte a una radic e comune, ma la radice ottenuta può anche non essere una parole reale del dizionario.
Una pipeline potrebbe essere:
testo -> tokenizazione -> stopwords removal -> stemming -> tf-idf -> classificatore

Lo stem è un processo euristico che taglia la fine delle parole nella speranza di raggiungere una forma base comune. A differenza della lemmatizzazione, che vedremo in seguito, lo stamming non usa un dizionario ma si basa su regole rigide di trasformazione delle stringhe.
L'obbiettivo prima ìrio è mappare varianti della stessa parola, come 'corressi', 'corriamo' o 'correre', verso un unico token rapresentativo. Questo riduce la sparsità del vocabolario e permette al modello di trattare concetti simili come un'unica enticà numerica

Ma perchè dovremmo voler tagvliare le parole?
il motivo è l'efficienza

In italiano un verbo può avere decine in flessioni diverse, per una rete neurale ogni flessione è potenzialmente un nuovo termine da imparare.
Riducendo tutto a una forma base, compriamo il vocabolario.
Matematicamente stiamo mappando un insieme vasto verso un insieme di radici molto più compatto, questo ha rivoluzionato il modo in cui cerchiamo informazioni.

Senza lo stemming i motori di ricerca sarebbero molto meno efficienti. Se cerchi giardiniere, probabilmente vuoi vedere i risultati anche per gardinaggio, e giardini, questo migliora la recall del sistema.
Poichè l'algoritmo di stemming è puramente algoritmico e non richiede database linguistici, può essere applicato in tempo reale durante l'indicizzazione di milioni di documenti.
Raggruppando le varianti, la matrice termine-documento diventa molto più compatta, ottimizzando lo storage dei modelli statistici. Inoltre rendiamo tutto più veloce accellerando l'addestramento dei modelli

Ma come fa un algoritmo dove tagliare senza sbagliare
Approfondimento: Teoria dei Suffissi
Scomposizone della parola
Ogni parola può essere vista come la concatenazione di una radice semantica e un affisso grammaticale. Lo stemming opera rimuovendo ricorsivamente questi affissi basandosi su lunghezze minime della radice residua. 
Sia 'w' una parola compsta da una radice 's' e un suffisso 'x'. Lo stemmer tenta di isolare 's' eliminando 'x' in base a regole di pattern matching

** Porter Stemmer **

è uno degli algoritmi di stemming più famosi. Fu proposto da Martin Porter (1980) e nasce principalmente per l'inglese.
Il suo approccio è basato su regole che eliminano o trasformano suffissi
Per esempio in inglese:
connected, connecting, connectin, connections vengono progressivamente ridotte secondo regole linguistiche e morfologiche.
Non consulta un dizionario, applica regole del tipo:
se termina in ing - prova a rimuovere ing
se termina in ed - prova a rimuovere ed
se termina in ization - trasformazione specifica

L'algoritmo lavora ad ondate
La prima fase è dedicata ai plurali ed alle desinenze verbali più ovvie (es. forme progressive ing)
Nelle fasi centrali vengono rimossi suffissi più complessi come 'ational' o 'fulness' riducnedoli a forme più semplici come 'ate' o 'ful'
L'ultima fase, di pulizia finale per sistemare le doppie consonanti e normalizzare ulteriormente la radice.

Porte Stemmer è veloce, semplice ed intuitivo ma puoi fare overstemming, cioè unire parole che in realtà hanno significati diversi.
Oppure understemming, cioè non unire parole che invece dovrebbero appartenere alla stessa famiglia
E soprattutto non è la scelta naturale per la lingua italiana

** Snowball **

Qui entra in gioco Snowball
Snowball è un'evoluzione del lavoro di Poter  ed è anche il nome del linguaggio/framework creato proprio per definire algoritmi di stemming, spesso chiamato anche Porter2
Snowball applica regolel specifiche della ingua.
Per l'italiano deve gestire suffissi e forme come: mente, zione, azini, ando, endo, ato, ito
Quindi non usa semplicemente le regole inglesi anche per l'italiano

Il vantaggio del stemming è che si riduce il vocabolario, questo rende modelli come tf-idf più compatti
Lo stemming lavora sulla struttura delle parole e non ha la comprensione grammaticale
Con la lemmatizzazione cerco la forma base linguisticamente corretta
quindi parole come sono, ero, sarò possono essere ricondotte a essere

Best Practies
- In italiano, preferite sempre Snowball rispetto a Potter, poichè quest'ultimo è ottimizzato quasi esclusivamente per l'inglse
- In ambiti medici o legali, lo stemming può essere troppo distruttivo. Valutare sempre se lemmatizzazione sia un'opzione migliore
- Verifa sempre l'impatto dello stemming sulle prestaszioni del classificatore finale: non sempre rimuovere morfologia aiuta l'accuratezza.

Compromesso Bias-Varianza
Riducendo il numero di termini riduciamo la varianza, il modello è meno confuso da tanti termini, ma aumentiamo i bias.
Lo stemming riduce la varianza del modello diminuendo il numero di feature (token), ma può introdurre bias se l'over-stemming confonde concetti diversi.
Dobbiamo bilanciare la capacità di generalizzazione con il rischio di perdere distinzioni semantiche cruciali.

Mentre la lemmatizzazione lavora sul significato delle parole
Per questo la lemmatizzazione è linguisticamente più sofisticata

Con SpaCy si preferisce fare lemmattizzazione che stemming, perchè spaCy ha informazioni grammaticali più ricche 

Come per le stopword la lemmatizzazione e lo stemming sono più frequenti con NLP mentre normalmente non sono fatte per Trasformer e LLM.
BERT o QWebb o Llama  sono modelli pre-addestrati su linguaggi naturale e il loro tokenizer gestisce già la rappresentazione di parole

In [1]:
import keras
import nltk
from nltk.stem import PorterStemmer, SnowballStemmer
from typing import List

# Download delle risorse NLTK necessarie
nltk.download('punkt')

def compare_stemmers(words: List[str], language: str = 'italian'):
    """
    Confronta l'efficacia di Porter e Snowball su una lista di termini.
    
    Teoria: Lo stemming è una 'regressione' morfologica. Riduce la varianza 
    aumentando però il rischio di bias (over-stemming).
    """
    
    # Inizializzazione degli stemmer
    # Porter: Il pioniere, basato su regole rigide per l'inglese.
    porter = PorterStemmer()
    
    # Snowball: Chiamato anche 'Porter2', più efficiente e multilingua.
    # Teoria: Snowball utilizza un linguaggio di programmazione specifico per 
    # descrivere algoritmi di stemming.
    snowball = SnowballStemmer(language=language)
    
    print(f"{'Parola Originale':<20} | {'Porter':<15} | {'Snowball (IT)':<15}")
    print("-" * 55)
    
    for word in words:
        p_stem = porter.stem(word)
        s_stem = snowball.stem(word)
        
        # Nota: Porter su parole italiane produrrà risultati spesso assurdi
        # perché cerca pattern morfologici inglesi (es: 'ing', 'ed', 's').
        print(f"{word:<20} | {p_stem:<15} | {s_stem:<15}")

# Dataset di test: Varianti verbali e potenziali casi di over-stemming
test_words = [
    "correre", "corriamo", "corressi",  # Varianti dello stesso verbo
    "università", "universo",           # Rischio over-stemming (radice comune?)
    "fiori", "fioraio", "fioritura",    # Famiglia semantica
    "andato", "andante"                 # Morfologia flessiva
]

print("--- ANALISI COMPARATIVA DELLO STEMMING ---")
compare_stemmers(test_words)


--- ANALISI COMPARATIVA DELLO STEMMING ---
Parola Originale     | Porter          | Snowball (IT)  
-------------------------------------------------------
correre              | correr          | corr           
corriamo             | corriamo        | corr           
corressi             | corressi        | corress        
università           | università      | univers        
universo             | universo        | univers        
fiori                | fiori           | fior           
fioraio              | fioraio         | fiorai         
fioritura            | fioritura       | fioritur       
andato               | andato          | andat          
andante              | andant          | andant         


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barbara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
